# Worldwide Earthquake Events API - Gold Layer Processing

In [1]:
from pyspark.sql.functions import when, col, udf, lit
from pyspark.sql.types import StringType
# ensure the below library is installed on your fabric environment
import reverse_geocoder as rg # new environment is created for this library

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 5, Finished, Available, Finished, False)

In [2]:
###Remove/Comment this cell

from datetime import date, timedelta

#start_date = date.today() - timedelta(25)
#end_date = date.today() - timedelta(1)

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 6, Finished, Available, Finished, False)

In [3]:
#df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)
from pyspark.sql.functions import col, lit
#This includes the start date and stops at midnight on the end date.
df = spark.read.table("earthquake_events_silver_v2").filter(
    (col("time") >= lit(start_date).cast("timestamp")) &
    (col("time") < lit(end_date).cast("timestamp"))
)

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 7, Finished, Available, Finished, False)

In [4]:
display(df)

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7169d35-b285-4292-9ec7-f39e789cbac4)

In [5]:
# retrieve country code
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_details(48.8588443, 2.2943506)
    'FR'
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get('cc')

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 9, Finished, Available, Finished, False)

In [6]:
# registering the udfs so they can be used on spark dataframes
# user define funtion is getting registrate
get_country_code_udf = udf(get_country_code, StringType())

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 10, Finished, Available, Finished, False)

In [7]:
# adding country_code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 11, Finished, Available, Finished, False)

In [9]:
# adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 13, Finished, Available, Finished, False)

In [10]:
display(df.limit(10))

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5aa2a6eb-0e1f-40ee-a308-0c07530cce6f)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number

# New Gold table name.
gold_table = "earthquake_events_gold_v2"

# Rank each earthquake's versions from newest to oldest.
window = Window.partitionBy("id").orderBy(
    col("updated").desc_nulls_last()
)

# Keep one record per earthquake ID in this batch.
incoming = (
    df_with_location_sig_class
    .filter(col("id").isNotNull())
    .withColumn("_row_number", row_number().over(window))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

if not spark.catalog.tableExists(gold_table):
    # First run: create the table using the cleaned batch.
    incoming.write.format("delta").saveAsTable(gold_table)

else:
    # Later runs: update existing events and insert new events.
    gold = DeltaTable.forName(spark, gold_table)

    (
        gold.alias("target")
        .merge(
            incoming.alias("source"),
            "target.id = source.id"
        )
        .whenMatchedUpdateAll(
            condition="""
                source.updated >= target.updated
                OR target.updated IS NULL
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

print(f"Gold processing completed: {gold_table}")

In [11]:
# appending the data to the gold table
#df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold_v2')

StatementMeta(, 3de9846f-5ace-42e3-99ab-f5767329c753, 15, Finished, Available, Finished, False)